In [9]:
import pandas as pd
import re
from sqlalchemy import create_engine
import urllibimport osfrom dotenv import load_dotenvload_dotenv()

In [10]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={os.getenv('MIRROR_DB_SERVER')};"
    "DATABASE=PROVENZAL;"
    f"UID={os.getenv('MIRROR_DB_USER')};"
    f"PWD={os.getenv('MIRROR_DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [11]:
query = """
SELECT DISTINCT c.CLICodigo, 
    c.CLINombres, 
    c.CLIApellidos,
    c.CLITelefonoCasa,
    c.CLICelular,
    c.CLIEmailPrincipal
    --c.CLITipoDocumento
FROM Clientes c
"""

# Ejecutar y cargar en DataFrame
df = pd.read_sql(query, engine)

In [13]:
#Limpiar Celular
def limpiar_celular(cel):
    if not cel:
        return ""
    cel = str(cel).strip().replace(" ", "").replace("-", "").replace("(", "").replace(")", "")
    if cel.startswith("+57"):
        cel = cel[3:]
    elif cel.startswith("57") and len(cel) > 10:
        cel = cel[2:]
    return cel

df['CLICelular'] = df['CLICelular'].apply(limpiar_celular)

#Validar celular
def es_celular_valido(cel):
    return bool(re.fullmatch(r"3\d{9}", str(cel)))

df['CelularValido'] = df['CLICelular'].apply(es_celular_valido)

#Limpiar y validar email
df['CLIEmailPrincipal'] = df['CLIEmailPrincipal'].apply(lambda x: str(x).strip())

def es_email_valido(email):
    email = str(email).strip()

    # Si está vacío
    if not email:
        return False
    
    # # Palabras prohibidas
    # palabras_no_permitidas = ["negad", "pendi"]
    # if any(palabra in email for palabra in palabras_no_permitidas):
    #     return False
    
    # Validación de formato
    return bool(re.fullmatch(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df['EmailValido'] = df['CLIEmailPrincipal'].apply(es_email_valido)

In [14]:
def limpiar_codigo(codigo):
    codigo = str(codigo).strip()     
    codigo = re.sub(r'^C\s*', '', codigo, flags=re.IGNORECASE)
    codigo = codigo.replace(' ', '')
    return codigo

df.insert(1, 'Documento', df['CLICodigo'].apply(limpiar_codigo))

df

,CLICodigo,Documento,CLINombres,CLIApellidos,CLITelefonoCasa,CLICelular,CLIEmailPrincipal,CelularValido,EmailValido
0,1013644206,1013644206,None,None,None,1111111111,NEGADO@PROVENZAL.NET,False,True
1,1014241430,1014241430,None,None,None,1111111111,NEGADO@PROVENZAL.NET,False,True
2,1015429541,1015429541,DANIELA,ORJUELA,11111111111,,,False,False
3,1018504327,1018504327,ALBERTO,CURE,3126605455,3126605455,betocure98@hotmail.com,True,True
4,1019024338,1019024338,MONICA PAOLA,GARZON GIL,,3144874565,MONISOFIA2014@GMAIL.COM,True,True
...,...,...,...,...,...,...,...,...,...
383199,CYC436201,YC436201,MAURICIO,VELLOSO SAMPAIO,3102642403,3102642403,MAURICIO.SAMPAIO@OUTLOOK.COM,True,True
383200,CYC498761,YC498761,MARIA,MANGA,1111111,1111111111,MARILUMANGA62@GMAIL.COM,False,True
383201,CYC6047662,YC6047662,SARA,BRESSAN,1111111,1111111111,NEGADO@PROVENZAL.NET,False,True
383202,CYI024605,YI024605,GERUSA,TEIXEIRA,1111111111,1111111111,NEGADO@PROVENZAL.NET,False,True


In [28]:
# # Detectar duplicados
# cel_duplicados = df['CLICelular'].duplicated(keep=False)
# email_duplicados = df['CLIEmailPrincipal'].duplicated(keep=False)

# # Marcar celulares duplicados como no válidos
# df.loc[cel_duplicados, 'CelularValido'] = False

# # Marcar correos duplicados como no válidos
# df.loc[email_duplicados, 'EmailValido'] = False

In [15]:
#Agregar columna contacto
def determinar_contacto(row):
    if row['CelularValido'] and row['EmailValido']:
        return "Cel + Email"
    elif row['CelularValido']:
        return "Cel"
    elif row['EmailValido']:
        return "Email"
    else:
        return "Falso"
    
df['Contacto'] = df.apply(determinar_contacto, axis=1)

In [16]:
# # Exportar todos los registros con validaciones incluidas
df.to_excel("email_validos2.xlsx", index=False)